In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score

features = pd.read_csv('../data/features.csv')

# Drop non-feature columns
drop_cols = ['account_id','churned','district_name','region']
X = features.drop(columns=drop_cols)
y = features['churned']

X = pd.get_dummies(X, columns=['frequency'], drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape, y_train.mean(), y_test.mean())

(3245, 28) (812, 28) 0.10015408320493066 0.09975369458128079


In [9]:
for col in ['unemp_95', 'crimes_95']:
    X_train[col] = pd.to_numeric(X_train[col].replace('?', np.nan))
    X_test[col] = pd.to_numeric(X_test[col].replace('?', np.nan))
    median_val = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_val)
    X_test[col] = X_test[col].fillna(median_val)

# Re-verify
for col in X_train.columns:
    if X_train[col].astype(str).eq('?').any():
        print('still bad:', col)
print('done')

done


In [10]:
import mlflow

mlflow.set_experiment("berka_churn")

results = {}

# 1. Dummy baseline
with mlflow.start_run(run_name="00_dummy"):
    dummy = DummyClassifier(strategy='most_frequent')
    dummy.fit(X_train, y_train)
    auc = roc_auc_score(y_test, dummy.predict_proba(X_test)[:,1])
    mlflow.log_metric("roc_auc", auc)
    results['dummy'] = auc

# 2. Logistic Regression
with mlflow.start_run(run_name="01_logistic_regression"):
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    logreg = LogisticRegression(max_iter=1000)
    logreg.fit(X_train_s, y_train)
    auc = roc_auc_score(y_test, logreg.predict_proba(X_test_s)[:,1])
    mlflow.log_metric("roc_auc", auc)
    results['logreg'] = auc

# 3. Random Forest
with mlflow.start_run(run_name="02_random_forest"):
    rf = RandomForestClassifier(n_estimators=200, random_state=42)
    rf.fit(X_train, y_train)
    auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:,1])
    mlflow.log_metric("roc_auc", auc)
    results['rf'] = auc

print(results)

{'dummy': 0.5, 'logreg': 0.6929624563003496, 'rf': 0.7410531826856495}


In [13]:
from xgboost import XGBClassifier

with mlflow.start_run(run_name="03_xgboost_full"):
    xgb = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42, eval_metric='logloss')
    xgb.fit(X_train, y_train)
    auc = roc_auc_score(y_test, xgb.predict_proba(X_test)[:,1])
    mlflow.log_metric("roc_auc", auc)
    results['xgb_full'] = auc

# Leakage sensitivity: does 'has_card' (card timing could leak) change the result?
leak_cols = ['has_card']
X_train_sub = X_train.drop(columns=leak_cols)
X_test_sub = X_test.drop(columns=leak_cols)

with mlflow.start_run(run_name="04_xgboost_subset_leaksensitivity"):
    xgb_sub = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42, eval_metric='logloss')
    xgb_sub.fit(X_train_sub, y_train)
    auc = roc_auc_score(y_test, xgb_sub.predict_proba(X_test_sub)[:,1])
    mlflow.log_metric("roc_auc", auc)
    results['xgb_subset'] = auc

print(results)

{'dummy': 0.5, 'logreg': 0.6929624563003496, 'rf': 0.7410531826856495, 'xgb_full': 0.7451318167232441, 'xgb_subset': 0.7495904477208627}


In [14]:
import joblib

final_model = xgb  # xgb_full — going with the full feature set since subset didn't meaningfully help
joblib.dump(final_model, '../models/best_model.joblib')

importances = pd.Series(final_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importances.head(10))

tx_amount_std                   0.070937
balance_last                    0.064886
n_orders                        0.051205
tenure_days                     0.048400
balance_mean                    0.047691
frequency_POPLATEK PO OBRATU    0.047006
balance_min                     0.043530
tx_amount_mean                  0.042585
crimes_96                       0.038696
n_entrepreneurs_per1000         0.038597
dtype: float32


In [16]:
import os
os.makedirs('../reports', exist_ok=True)

importances.to_csv('../reports/feature_importance.csv')